In [4]:
import sys
print(sys.executable)

/Users/teslim/TeslimWorkSpace/PorfolioRepo/portfolio-25-llm-agentic-rag/.venv/bin/python


In [5]:
from dotenv import load_dotenv
import openai

load_dotenv()
openai_client = openai.OpenAI()

# 1. Start with a minimal RAG baseline

Before retrieval, we need to see what an LLM can and cannot do with a plain prompt. This section sends a question directly to the model, then adds a small piece of context manually.

**Interactive checkpoint:** run each cell, compare the answers, and ask yourself: *What information came from the model's prior knowledge, and what information came from the context we supplied?*

In [6]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [7]:
llm("Hey, what's up?")

"Hello! I'm here and ready to help you with anything you need. What's on your mind?"

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Whether you can join a course that's already in progress depends on the specific course's policies and structure. Some courses might allow late enrollments, especially if they are self-paced or have flexible schedules, while others may have strict deadlines or limited seating. 

I recommend checking the course's official website or contacting the course administrator for more information on enrollment options.


In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# 2. Load the FAQ knowledge base

The FAQ dataset is the knowledge base for this project. We first fetch the course catalogue, inspect its shape, then follow each course path to collect the individual FAQ records.

**What to notice:** the catalogue describes where each course's data lives; it does not contain every question and answer itself. The next cells turn that catalogue into one combined list called `documents`.

In [14]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

## Inspect the source catalogue

This request downloads a JSON catalogue of available courses and parses it into Python objects.

- `requests.get(...)` performs the HTTP request.
- `.json()` converts the response body into dictionaries and lists.
- Inspecting `type`, `len`, and the first item is a useful habit before writing the ingestion loop.

**Try changing the inspection cells:** print a different course record and compare its `path`, `course`, and `questions_count` fields.

In [15]:
# 1) Check top-level type
print(type(courses_raw))

<class 'list'>


In [ ]:
print(len(courses_raw))      # how many items
print(type(courses_raw[0]))  # type of first item (usually dict)
print(courses_raw[0])    

6
<class 'dict'>
{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}


In [ ]:
print(courses_raw)  # Print the first course for inspection

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 103}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


## Download and combine the course FAQs

Each catalogue item points to a separate JSON file. The loop downloads those files one by one and extends `documents` with their records.

This is the ingestion stage:

```text
course catalogue -> course FAQ URLs -> individual FAQ records -> one knowledge base
```

`raise_for_status()` makes failures visible immediately instead of silently adding incomplete data. In a production ingestion job, you would also consider timeouts, retries, logging, and caching.

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1368

This block builds one combined FAQ dataset by downloading each course’s FAQ JSON and merging all entries into a single list.

1. `documents = []`  
Creates an empty list to store all FAQ items from all courses.

2. `url_prefix = "https://datatalks.club/faq"`  
Defines the base URL used to build each course-specific FAQ endpoint.

3. `for course in courses_raw:`  
Loops through each course record in `courses_raw` (fetched earlier from `courses.json`).

4. `course_url = f"{url_prefix}{course['path']}"`  
Builds the full URL for that course’s FAQ JSON using its `path`.

5. `course_response = requests.get(course_url)`  
Sends a GET request to fetch that course’s FAQ content.

6. `course_response.raise_for_status()`  
Throws an error if the request failed (e.g., 404/500), which helps catch bad URLs or server issues early.

7. `course_data = course_response.json()`  
Parses the response body into Python data (typically a list of FAQ dicts).

8. `documents.extend(course_data)`  
Appends all FAQ items from this course into the master `documents` list.

9. `len(documents)`  
Returns how many total FAQ entries were collected across all courses.

In short: it aggregates many per-course FAQ files into one searchable list.


Each entry has:  

`id` - unique identifier.  
`course` - course slug (e.g., machine-learning-zoomcamp).  
`section` - which section of the course.  
`question` - the FAQ question.  
`answer` - the FAQ answer.  

In [ ]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
documents[400]

{'id': 'a01ea88b00',
 'course': 'data-engineering-zoomcamp',
 'section': 'Workshop 1 - dlthub',
 'question': 'REST API pagination should start at page 1 and stop when the API returns an empty list',
 'answer': "Problem:\n- Some REST APIs paginate results using a page parameter where valid pages start at 1. Using page=0 can return an empty list, leading to incomplete ingestion.\n\nBest practice:\n- Start from page=1\n- Continuously request subsequent pages until the API returns an empty list\n- Do not rely on page=0 to fetch data; always test the API behaviour manually before implementing pagination logic\n\nImplementation example (Python):\n```python\nimport requests\n\ndef fetch_all_pages(base_url, start_page=1, per_page=None):\n    page = start_page\n    all_items = []\n    while True:\n        params = {'page': page}\n        if per_page is not None:\n            params['per_page'] = per_page\n        resp = requests.get(base_url, params=params)\n        resp.raise_for_status()\n   

Viewing each FAQ entry’s structure helps us understand what fields are available for retrieval and generation tasks. Each entry typically contains a `question`, `answer`, and `course` field, which we can use to build our RAG system.

In [ ]:
documents[400]['question']

'REST API pagination should start at page 1 and stop when the API returns an empty list'

In [ ]:
documents[400]['course']

'data-engineering-zoomcamp'

In [ ]:
documents[400]['answer']

"Problem:\n- Some REST APIs paginate results using a page parameter where valid pages start at 1. Using page=0 can return an empty list, leading to incomplete ingestion.\n\nBest practice:\n- Start from page=1\n- Continuously request subsequent pages until the API returns an empty list\n- Do not rely on page=0 to fetch data; always test the API behaviour manually before implementing pagination logic\n\nImplementation example (Python):\n```python\nimport requests\n\ndef fetch_all_pages(base_url, start_page=1, per_page=None):\n    page = start_page\n    all_items = []\n    while True:\n        params = {'page': page}\n        if per_page is not None:\n            params['per_page'] = per_page\n        resp = requests.get(base_url, params=params)\n        resp.raise_for_status()\n        data = resp.json()\n        if not data:\n            break\n        all_items.extend(data)\n        page += 1\n    return all_items\n```\n\nManual testing steps:\n- Call the endpoint with page=1 and verif

## Inspect one document before indexing

A document is a structured FAQ record, not just a string. The fields have different jobs:

- `question` and `answer` contain searchable text;
- `course` supports exact filtering;
- `section` adds useful context and can influence ranking;
- `id` identifies the record.

**Interactive checkpoint:** inspect different positions in `documents`, then predict which fields should be searched and which field should be used as a filter.

Each course has a slug - a short identifier used in URLs. For example, machine-learning-zoomcamp, data-engineering-zoomcamp, etc. We'll use these slugs for filtering in search.

Using this data
In the RAG pipeline, this dataset is our knowledge base:

We index all the documents (the search step)
- When a student asks a question, we search the index
- The search returns the most relevant FAQ entries
- We give those entries to the LLM as context
- The LLM generates an answer based on the context
- The `question` and `answer` fields contain the text we'll search through. The `course` field lets us filter by course. For example, if a student asks about the data engineering course, we skip results from the ML course. The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

# 3. Build a keyword search index with `minsearch`

Retrieval is the bridge between the user's question and the LLM's context. Instead of sending the entire FAQ collection to the model, we create an index that can quickly return the most relevant records.

This notebook uses `minsearch`, an in-memory keyword search library. It is deliberately simple and useful for learning; later, the SQLite notebook replaces this temporary index with a persistent search index.

## Indexing and Searching the FAQ Dataset

We already have the `documents` list from the previous section (~1,100 entries). Now we build a searchable index on top of it.

📺 [Video reference](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=7)

---

### Why search instead of sending everything to the LLM?

Passing all 1,100 documents into every LLM call would be:

- **Expensive** — more tokens = higher API cost
- **Slow** — large prompts increase latency
- **Noisy** — too much irrelevant context degrades answer quality

Search first retrieves a small set of high-relevance documents, and only those are sent to the LLM as context.

---

### Why `minsearch` instead of Elasticsearch or Solr?

Production search engines like Apache Lucene, Elasticsearch, and Solr are powerful but heavy — Elasticsearch typically requires running a separate Docker container, which is not feasible in environments like Google Colab.

`minsearch` is a lightweight, in-memory keyword search library built for exactly this use case: small datasets, learning environments, and prototyping. It uses the same terminology as Elasticsearch deliberately, so skills transfer directly to larger systems.

---

### Core concepts

#### Text fields
Used for full-text search and relevance ranking. The engine tokenises the field values, lowercases them, removes stop words, and scores matches against your query.

In this notebook, `question`, `section`, and `answer` are text fields — all can contain words relevant to a user's query.

#### Keyword fields
Used for exact matching and hard filtering. Equivalent to a SQL `WHERE` clause:

```sql
SELECT * FROM index WHERE course = 'llm-zoomcamp'
```

The `course` field is a keyword field so we can restrict search to one specific course (e.g. LLM Zoomcamp) and exclude entries from MLOps or Data Engineering Zoomcamp.

#### Boosting
Boosting assigns different importance weights to fields when ranking results. Fields with higher boost scores contribute more to the final ranking.

| Field | Boost | Reason |
|-------|-------|--------|
| `question` | 2.0 | A keyword match in the question is the strongest relevance signal |
| `answer` | 1.0 | Default weight |
| `section` | 0.5 | Less informative for ranking, but still useful |

**Example:** if the word *"certificate"* appears in both a question and an answer, the result where it appears in the question ranks higher because the question field has boost 2.0.

In code:
```python
boost_dict = {"question": 2.0, "section": 0.5}
```

---

### How search fits into the RAG pipeline

The search function becomes the first step in the RAG flow:

1. User submits a question
2. `search()` retrieves the top 5 most relevant FAQ entries (filtered by course)
3. Those entries are passed to the LLM as context
4. The LLM generates an answer grounded in the retrieved documents

This keeps the prompt small, focused, and relevant.

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

## Explore the `minsearch` API

The next cells are a small investigation exercise. `dir()` shows the names exposed by the module, while `help()` and `inspect.getsource()` let us examine the index and its search method.

You do not need to memorise every method. The useful habit is knowing how to inspect an unfamiliar library before building on it.

**Interactive checkpoint:** look for the `fit` and `search` methods. Predict what each one does, then compare your prediction with the documentation and source output.

In [ ]:
import minsearch
dir(minsearch)          # all attributes and methods

['AppendableIndex',
 'DEFAULT_ENGLISH_STOP_WORDS',
 'Highlighter',
 'Index',
 'STEMMERS',
 'Tokenizer',
 'VectorSearch',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'append',
 'filters',
 'get_stemmer',
 'highlighter',
 'lancaster_stemmer',
 'minsearch',
 'porter_stemmer',
 'snowball_stemmer',
 'stemmers',
 'tokenizer',
 'vector']

In [ ]:
[x for x in dir(minsearch) if not x.startswith("_")]

['AppendableIndex',
 'DEFAULT_ENGLISH_STOP_WORDS',
 'Highlighter',
 'Index',
 'STEMMERS',
 'Tokenizer',
 'VectorSearch',
 'append',
 'filters',
 'get_stemmer',
 'highlighter',
 'lancaster_stemmer',
 'minsearch',
 'porter_stemmer',
 'snowball_stemmer',
 'stemmers',
 'tokenizer',
 'vector']

In [ ]:
help(minsearch.Index)         # full docstring for the Index class

In [ ]:
help(minsearch.filters)  # just the search method

In [ ]:
help(minsearch.Index.search)  # just the search method

In [ ]:
import inspect
print(inspect.getsource(index.search))

Having a clear understanding of the `minsearch` module will allow us to effectively build and query our index, ensuring that we can retrieve the most relevant documents for our RAG system.

## 4. Run the first filtered search

Now we use the index with four retrieval controls:

- the user's `question`;
- boosted fields, so matches in `question` matter more than matches in `section`;
- a course filter, so unrelated courses are excluded;
- `num_results=5`, which limits the context size.

Before running the cell, predict which FAQ question should appear first. Then inspect `search_results` and compare the returned ranking with your prediction.

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Experiment: change only the course filter

The query stays the same, but the filter changes from `llm-zoomcamp` to `machine-learning-zoomcamp`. The results should now come from the selected course.

**Try this interactively:** change the query to `"How do I submit homework?"`, run both filter versions, and note how the course filter changes the knowledge available to the answer.

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "machine-learning-zoomcamp"},
    num_results=5
)

search_results

[{'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': 'e0a95572a6',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How do I join Slack if the invite email didn’t arrive?',
  'answer': 'Go to DataTalks.Club, request a Slack invite, or use the manual request form (processed daily). After joining, browse channels and join **#course-ml-zoomcamp**.'},
 {'

## 5. Make filtering reusable

A course filter is part of the retrieval policy, so it belongs inside a reusable function rather than being copied into every query. The next cells isolate that behavior and make the course an explicit parameter.

Sometimes you want to restrict the search to a specific course.

minsearch supports keyword filtering:

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

print(results)

[{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}, {'id': 'c842475338', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Homework: Just found this course, can I still submit homeworks?', 'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "Open/Submittable":

This only returns documents from the MLOps Zoomcamp. Try a few different queries and courses to get a feel for the results.

In [ ]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

#### Wrapping it in a function

Let's wrap the search in a search function - the first component of our RAG pipeline:

In [ ]:
question = "I just discovered the course. Can I join now?"

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

### Explaining the `search()` function

This function is a small wrapper around `index.search(...)`. Its purpose is to keep the retrieval logic in one place so the RAG pipeline can call a clean, reusable function instead of repeating the full search configuration each time.

#### Function definition

`def search(question, course="llm-zoomcamp"):`

- `question` is the user's query
- `course="llm-zoomcamp"` sets the default course filter
- If no course is passed, the function searches within the LLM Zoomcamp documents

So both of these are valid:

```python
search("Can I still join?")
search("Can I still join?", course="mlops-zoomcamp")
```

#### Boosting

```python
boost_dict = {"question": 2.0, "section": 0.5}
```

This dictionary controls how strongly different fields affect ranking:

- `question: 2.0` means matches in the FAQ question are treated as highly important
- `section: 0.5` means matches in the section name are weaker signals
- `answer` is not explicitly listed, so it uses the default weight

This setup makes sense because the wording of a user's query is usually closest to the FAQ `question` field.

#### Filtering

```python
filter_dict = {"course": course}
```

This restricts the search to documents from one course only. Without this filter, results could come from other Zoomcamp courses and add noise to the answer.

For example, if `course="llm-zoomcamp"`, only documents whose `course` field equals `"llm-zoomcamp"` are returned.

#### Running the search

```python
return index.search(
    question,
    boost_dict=boost_dict,
    filter_dict=filter_dict,
    num_results=5
)
```

This sends the query to the index and returns the top 5 matching documents.

- `question` is the search query
- `boost_dict` adjusts field importance during ranking
- `filter_dict` limits results to the chosen course
- `num_results=5` keeps the returned context small and manageable for the LLM

#### Why this function is useful in RAG

This function becomes the retrieval step in the RAG pipeline:

1. Take the user's question
2. Search the indexed FAQ documents
3. Return the most relevant results
4. Pass those results to the LLM as context

Wrapping the logic in a function makes the code easier to reuse, test, and modify later. For example, if you want to change the number of retrieved documents or adjust the boosts, you only need to update this one function.

In [ ]:
search_results = search(question)

#### What `search_results = search(question)` does

This line runs the custom `search()` function using the current value stored in `question`. The returned results are saved in the variable `search_results`.

#### Step by step

1. It reads the value of `question`
   Earlier in the notebook, `question` was defined as the user's query, for example:

```python
question = "I just discovered the course. Can I join now?"
```

2. It calls `search(question)`
   This runs the wrapper function we defined earlier. Inside that function:

- `boost_dict = {"question": 2.0, "section": 0.5}` adjusts how strongly different fields affect ranking
- `filter_dict = {"course": "llm-zoomcamp"}` limits the search to LLM Zoomcamp documents by default
- `num_results=5` returns the top 5 most relevant documents

3. It stores the results in `search_results`
   After the function finishes, `search_results` contains a list of the best matching FAQ documents for the question.

#### In plain English

This line means:

- take the user's question
- search the FAQ index
- keep only documents from the selected course
- rank the results using boosting
- return the top 5 matches
- store them in `search_results`

#### Why this matters in RAG

This is the retrieval step of the RAG pipeline. The documents stored in `search_results` are the pieces of context that will later be passed to the LLM to generate an answer.

#### Useful ways to inspect the results

```python
search_results
```

```python
[doc["question"] for doc in search_results]
```

```python
search_results[0]["answer"]
```

## Building the Prompt

Video: [Watch this lesson](https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In a RAG system, the LLM cannot read our FAQ documents automatically. It only sees what we send inside the prompt. That is why we need a prompt-building step after search.

### The idea in simple terms

The flow is:

1. The user asks a question
2. `search()` retrieves the most relevant FAQ entries
3. We turn those entries into one clean text block called `context`
4. We combine the user's question and the context into a prompt
5. We send that prompt to the LLM

So this section is the bridge between **retrieval** and **generation**.

### Why split the prompt into two parts?

We usually separate the prompt into:

- **Instructions**: fixed rules that tell the LLM how to behave
- **User prompt**: the changing part that contains the actual question and retrieved context

This split is useful because the instructions stay the same for every user, while the question and context change every time.

### What the instructions do

The instructions tell the model:

- answer using the provided context
- stay focused on the course FAQs
- say `I don't know.` if the answer is not in the retrieved documents

This helps reduce hallucinations and keeps the answer grounded in the dataset.

### What the user prompt does

The user prompt is the part we build dynamically for each question. It includes:

- the user's question
- the formatted FAQ entries returned by search

So the LLM receives both the task and the evidence it should use.

### Why `build_context()` is needed

`search_results` is a list of Python dictionaries. The LLM does not work directly with Python objects, so we convert the list into readable text.

Each result is turned into a small block like this:

```text
Section name
Q: question text
A: answer text
```

That makes the retrieved documents easier for the LLM to read and use.

### Why `build_prompt()` is needed

Once the context text is ready, we insert it into a prompt template together with the user's question. The final prompt is what gets passed to the LLM.

A good prompt matters because it controls how well the model uses the retrieved context. If the prompt is vague, the model may ignore the evidence. If the prompt is clear, the answer is more likely to stay grounded.

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with \"I don't know.\"
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### Explaining the code step by step

##### `INSTRUCTIONS`

```python
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with \"I don't know.\"
"""
```

This is the fixed instruction block for the LLM. It defines the model's role and tells it how to behave. The key idea is grounding: the model should answer from the provided context, not from guesses.

##### `USER_PROMPT_TEMPLATE`

```python
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""
```

This is a template with placeholders. Later, `{question}` will be replaced by the user's actual question, and `{context}` will be replaced by the retrieved FAQ text.

#### `build_context(search_results)`

```python
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
```

This function converts the retrieved documents into one readable string.

- `search_results` is a list of dictionaries
- `lines = []` starts an empty list for storing text lines
- `for doc in search_results:` loops through each retrieved FAQ document
- `doc["section"]` adds the section name
- `"Q: " + doc["question"]` adds the FAQ question
- `"A: " + doc["answer"]` adds the FAQ answer
- `lines.append("")` adds a blank line between documents

At the end, `"\n".join(lines)` joins all the lines into one text block, and `.strip()` removes extra whitespace at the beginning or end.

#### `build_prompt(question, search_results)`

```python
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
```

This function builds the final user prompt.

- It first calls `build_context(search_results)` to convert the retrieval results into text
- It then inserts `question` and `context` into the prompt template using `.format(...)`
- Finally, it returns the completed prompt as a clean string

#### Testing the prompt

```python
prompt = build_prompt(question, search_results)
print(prompt)
```

This creates the final prompt and prints it so you can inspect it before sending it to the LLM.

You should expect to see:

- the user's question at the top
- several retrieved FAQ entries below it
- each entry shown as section, question, and answer

### Why this section matters

This is the point where raw retrieval results become LLM-ready input. Search gives us relevant documents, but prompt building turns them into structured context the model can actually use. Without this step, the LLM would not see the retrieved FAQ information.

# 6. Build the final RAG response

Retrieval gives us evidence; the LLM turns that evidence into a readable answer. This section connects the search function, prompt construction, and model call into one end-to-end flow.

**Interactive checkpoint:** try one question that should be answered by the FAQ and one question that is unlikely to be covered. Check whether the instruction to say `I don't know.` is respected.

Video: [Watch this lesson](https://www.youtube.com/watch?v=KHePGkeFn54&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

This section explains the final part of the RAG pipeline: sending our built prompt to the LLM and getting an answer back.

### Big picture

So far we already have:

- `search(query)` to retrieve relevant FAQ documents
- `build_prompt(query, search_results)` to combine question + context

Now we add the LLM step, which does:

1. receive the prompt
2. generate an answer
3. return text we can show to the user

### Why this step matters

Without this step, we only have retrieved documents. The LLM is the component that turns those documents into a natural-language answer.

### Two ways to send input to OpenAI

You can send a single string prompt directly, or send a message history (roles + content).

- Single prompt: simple for quick tests
- Message history: better structure for real apps because it separates instructions from user content

In this notebook, we use message history for the final `llm()` function.

### What we will do in code

- Send a prompt with the Responses API
- Read the answer from `response.output_text`
- Inspect token usage and estimate request cost
- Create reusable `llm()` and `rag()` functions

This gives us a complete end-to-end RAG flow.

In [ ]:
# We already built `prompt` in the previous section.
## prompt = build_prompt(question, search_results)

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

# The easiest way to read the model answer
print(response.output_text)
print("")

# Optional: inspect token usage
print(response.usage)
print("")

# Example cost estimate for gpt-5.4-mini
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

print(f"Estimated cost: ${cost:.8f}")

Yes — you can still join now and start learning.

If you want a certificate, make sure you submit your project while submissions are still open, since certificates are only available during the live cohort.

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=523)

Estimated cost: $0.00055350


### Explaining the code above (line by line)

#### `openai_client.responses.create(...)`

```python
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)
```

- `model` selects which LLM to use
- `input=prompt` sends the text we built from question + retrieved context
- The returned `response` object contains answer text, usage stats, and metadata

#### `response.output_text`

This is the shortcut to get the final answer string directly.

You could also navigate low-level fields like `response.output[0].content[0].text`, but `response.output_text` is cleaner and easier to read.

#### `response.usage`

This shows token counts, such as:

- `input_tokens`
- `output_tokens`
- `total_tokens`

Token usage helps you monitor cost and optimize prompt size.

#### Cost calculation

```python
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000
```

These are per-token prices derived from per-million-token pricing. Then we multiply by usage counts to estimate the request cost.

This is useful when testing different prompt sizes, retrieval depths, or models.

### Message history format (recommended)

For better structure, we usually send two messages:

- `developer`: fixed instructions (`INSTRUCTIONS`)
- `user`: current question + context prompt

This keeps role guidance separate from the changing user content and is a good pattern for production RAG systems.

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


In [ ]:
# Try end-to-end RAG
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [ ]:
print(rag("How do I get a certificate?"))
print("")

You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.

Self-paced mode does not include certificates. Also, you need to peer-review 3 capstones during the course while the peer-review form is still open.



In [ ]:
print(rag("How much does the course cost?   "))

I don't know.


## RAG Helper

# 7. Move repeated logic into reusable modules

The notebook has now built the RAG pipeline step by step: retrieval, prompt construction, and LLM generation. That is useful for learning, but copying these blocks into every notebook makes changes harder to maintain.

The next stage moves the shared logic into:

- `ingestion.py`: loads FAQ data and builds the search index;
- `rag_pipeline.py`: contains the reusable search, prompt-building, and LLM workflow.

The notebooks can then focus on experiments while the implementation lives in tested, reusable Python modules. This is the transition from exploration to a maintainable project structure.

**Try it next:** open `02-reusable-rag-pipeline.ipynb` and compare how much setup disappears.

## 8. Continue with the reusable pipeline

Notebook 01 ends with the concepts in place: load data, build an index, retrieve relevant documents, build grounded context, and generate an answer.

Continue with [02-reusable-rag-pipeline.ipynb](02-reusable-rag-pipeline.ipynb) to move the repeated logic into `ingestion.py` and `rag_pipeline.py`.